# 02 - Data Cleaning

This notebook documents and reproduces the cleaning rules required by the RetailPulse brief.


## Cleaning policy
- Missing Customer ID is **not imputed** because a synthetic ID would fabricate longitudinal history.
- Missing Description is imputed as `Unknown Product Description`.
- Exact duplicates are removed.
- Cancellation-style invoices and non-positive transactions are excluded from the primary completed-sales table.
- IQR-capped helper fields (`Quantity_Capped`, `Price_Capped`) are generated for robust analytical/modeling use. Original Quantity/Price and primary Revenue are preserved.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from config import *
from src.data_loader import load_data
from src.cleaning import clean_transactions
raw=load_data("auto")
sales,deduped,summary=clean_transactions(raw)
summary


In [ ]:
print("Raw rows:", len(raw))
print("Deduplicated rows:", len(deduped))
print("Primary positive-sales rows:", len(sales))
print("Description imputation rows:", int(sales["DescriptionWasMissing"].sum()))
print("Quantity cap count:", int((sales["Quantity"] != sales["Quantity_Capped"]).sum()))
print("Price cap count:", int((sales["Price"] != sales["Price_Capped"]).sum()))
sales.to_csv(PROCESSED_DIR/"retail_cleaned.csv",index=False)


## Reproducibility and validation
This notebook should reproduce documented imputation, duplicate handling, exclusions, and IQR analytical outlier capping.
Run the preceding analysis cells from top to bottom after installing the project requirements. The final artifact checks below are intentionally lightweight so the notebook can also be used to verify an existing pipeline run.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'artifacts').exists(): ROOT = ROOT.parent
required = [ROOT / 'artifacts/metrics/cleaning_summary.json'
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f'Missing expected artifact(s): {missing}'
print('Validation passed for:', ', '.join(p.name for p in required))
